### 3.6.1 MNIST 数据集

#### MNIST 数据集

首先,为了导入父目录中的文件,进行相应的设定。然后,导入dataset/mnist.py中的load_mnist函数。最后,使用load_mnist函数,读入MNIST数据集。

用来读入 MNIST 图像的文件在本书提供的源代码的 dataset 目录下。并且,我们假定了这个 MNIST 数据集只能从 ch01、ch02、ch03、...、ch08 目录中使用,因此,使用时需要从父目录(dataset目录)中导入文件,为此需要添加sys.path.append(os.pardir) 语句。

In [1]:
import sys,os
sys.path.append(os.pardir) # 为了导入父目录中的文件而进行的设定 # pardir 是 Parent Directory（父目录/上一级目录）的缩写
from dataset.mnist import load_mnist # 从 dataset 文件夹下的 mnist.py 文件中，导入名为 load_mnist 的函数。

load_mnist 函数以“( 训练图像 , 训练标签 ),( 测试图像,测试标签 )”的形式返回读入的 MNIST 数据。

此外,还可以像 load_mnist(normalize=True, flatten=True, one_hot_label=False) 这样,设置3个参数。

- 第1个参数 normalize 设置是否将输入图像正规化为 0.0-1.0 的值。如果将该参数设置为 False,则输入图像的像素会保持原来的 0-255。
- 第2个参数 flatten 设置是否展开输入图像(变成一维数组)。如果将该参数设置为 False,则输入图像为 1 × 28 × 28 的三维数组;若设置为True,则输入图像会保存为由 784 个元素构成的一维数组。
- 第3个参数 one_hot_label 设置是否将标签保存为 onehot 表示(one-hot representation)。one-hot 表示是仅正确解标签为 1,其余皆为 0 的数组,就像 [0,0,1,0,0,0,0,0,0,0] 这样。当 one_hot_label 为 False 时,只是像 7、2 这样简单保存正确解标签;当 one_hot_label 为 True 时,标签则保存为 one-hot 表示。

In [3]:
# 第一次调用会花费几分钟 ......
(x_train,t_train),(x_test,t_test) = load_mnist(flatten = True, normalize = False)

Done
Done
Done
Done
Converting train-images-idx3-ubyte.gz to NumPy Array ...
Done
Converting train-labels-idx1-ubyte.gz to NumPy Array ...
Done
Converting t10k-images-idx3-ubyte.gz to NumPy Array ...
Done
Converting t10k-labels-idx1-ubyte.gz to NumPy Array ...
Done
Creating pickle file ...
Done!


In [4]:
# 输出各个数据的形状
print(x_train.shape)
print(t_train.shape)
print(x_test.shape)
print(t_test.shape)

(60000, 784)
(60000,)
(10000, 784)
(10000,)


#### 显示MNIST图像

In [5]:
### 第一部分：导入工具包 (Imports)
import sys, os
sys.path.append(os.pardir)
import numpy as np
from dataset.mnist import load_mnist
from PIL import Image

### 第二部分：定义图片显示函数 (Define Function)
def img_show(img):
    pil_image = Image.fromarray(np.uint8(img))
    # Image.fromarray(...)：把 NumPy 数组（一堆数字组成的矩阵）转换成 PIL 图像对象。
    # np.uint8(img)：把图片数据转换成 uint8 格式（8位无符号整数，范围在 0~255 之间）。这是计算机存储标准图片像素的最基本格式。
    pil_image.show()

### 第三部分：加载并提取数据 (Load Data)
(x_train,t_train),(x_test,t_test) = load_mnist(flatten = True, # flatten=True：意思是把原本28*28像素的二维图片“压扁”成一维的、包含 784个数字的扁平数组
                                              normalize = False) # normalize=False：保持像素值原本的样子（0~255）。如果设置为 True，像素值会被缩放到 0.0~1.0 之间
img = x_train[0] # 取出训练集里的第一张图片的数据
label = t_train[0] # 取出第一张图片对应的正确答案（标签）
print(label) # 5
# 打印这个标签。结果是 5，说明这第一张图片画的是数字 5。

### 第四部分：重塑形状与显示 (Reshape & Show)
print(img.shape) # (784,)
img = img.reshape(28,28) # 把图像的形状变成原来的尺寸
# 核心步骤！ 因为一维的一排数字是没有办法当成图片显示的，所以通过 .reshape(28, 28) 函数，把这 784 个数字重新重新排列成 $28 \times 28$ 的二维方阵（矩阵）。
print(img.shape) # (28,28)

img_show(img)

5
(784,)
(28, 28)


In [6]:
img_show(img)

### 3.6.2 神经网络的推理处理

In [7]:
def sigmoid(x):
    return 1/(1+np.exp(-x))

In [8]:
def softmax(a):
    c = np.max(a)
    exp_a = np.exp(a-c)
    sum_exp_a = np.sum(exp_a)
    y = exp_a / sum_exp_a
    return y

In [12]:
import pickle

In [10]:
### 第一部分：获取测试数据 (get_data)
def get_data():
    (x_train,t_train),(x_test,t_test) = load_mnist(normalize = True, flatten = True, one_hot_label = False)
    return x_test, t_test
    # normalize=True：归一化。把原本 0-255 的像素值缩放到 0.0-1.0 之间。这能让神经网络计算时更稳定、更高效。
    # flatten=True：扁平化。把28*28的二维图片压扁成包含 784 个数字的一维数组。
    # one_hot_label=False：不使用独热编码。这意味着标签（正确答案）直接返回普通的数字（比如 5、7），而不是返回一维数组（如 [0,0,0,0,0,1,0,0,0,0]）。
    # return x_test, t_test：由于这里是执行推理（测试）而不是训练，所以函数只返回用于测试的图片数据 x_test 和对应的正确标签 t_test。

### 第二部分：初始化网络参数 (init_network)
# init_network() 会读入保存在 pickle 文件 sample_weight.pkl 中的学习到的权重参数。
# 这个文件中以字典变量的形式保存了权重和偏置参数。剩余的 2 个函数,和前面介绍的代码实现基本相同,无需再解释。
def init_network():
    with open('sample_weight.pkl','rb') as f: 
        # 以二进制只读模式 ('rb') 打开一个名为 sample_weight.pkl 的文件。以“二进制、只读”模式打开文件 (Read Binary)。
        # 注：这是一个 Pickle 文件，里面保存的是作者提前用电脑训练好的、最完美的权重和偏置参数。
        network = pickle.load(f)
        # 利用 pickle.load 把文件里的内容反序列化，读取出来。读取后，network 变成了一个 Python 字典（Dictionary），里面存着网络各层的权重和偏置。
    return network

### 第三部分：前向传播预测 (predict)
def predict(network, x):
    W1, W2, W3 = network['W1'], network['W2'], network['W3']
    b1, b2, b3 = network['b1'], network['b2'], network['b3']
    a1 = np.dot(x,W1) + b1
    z1 = sigmoid(a1)
    a2 = np.dot(z1,W2) + b2
    z2 = sigmoid(a2)
    a3 = np.dot(z2,W3) + b3
    y = softmax(a3)
    return y

把测试集里的 10000 张图片一张一张地送进神经网络，让网络去猜（预测）每张图是什么数字。然后，统计网络一共猜对了多少张，最后计算出神经网络的“准确率 (Accuracy)”。

In [13]:
### 第一部分：准备工作 (Initialization): 获得 MNIST 数据集,生成网络
x, t = get_data()
network = init_network()

### 第二部分：核心循环比对 (The For-Loop)
accuracy_cnt = 0
# 变量赋值:定义一个名为 accuracy_cnt 的计数器，初始值为 0。它就像一个记分板，专门用来记录神经网络猜对了多少张图片。
for i in range(len(x)):
    # len(x) 的值是 10000（因为有 10000 张测试图）。这行代码启动了一个循环，变量 i 会从 0 一直变到 9999。这意味着接下来的步骤要重复执行 10000 次。
    y = predict(network, x[i])
    # 含义：x[i] 代表当前第 i 张图片的数据。把这单张图片和网络参数送入 predict 函数。
    # 返回的 y 是一个包含 10 个概率值的 NumPy 数组（因为输出层用了 Softmax 函数）。例如：[0.05, 0.01, 0.02, 0.02, 0.10, 0.75, 0.01, 0.02, 0.01, 0.01]。这代表网络认为这张图是 0 的概率是 5%，是 5 的概率是 75%……
    p = np.argmax(y) # 获取概率最高的元素的索引
    # np.argmax(y) 的功能是“找出数组中最大值的索引（位置）”。
    # 结合上面的例子，y 里面最大的是 0.75，它排在第 5 个位置（索引从 0 开始算：0, 1, 2, 3, 4, 5）。所以 p 的值就会变成 5。这就代表神经网络最终的预测结果（它的猜测）是 5。
    if p == t[i]: # 对答案  
        accuracy_cnt += 1

### 第三部分：计算并打印准确率 (Calculate Accuracy)
print('Accuracy' + str(float(accuracy_cnt / len(x))))

Accuracy0.9352


#### 3.6.3 批处理

输入数据和权重参数的‘形状’

In [15]:
x,_ = get_data()
network = init_network()
W1, W2, W3 = network['W1'], network['W2'], network['W3']

In [16]:
x.shape

(10000, 784)

In [17]:
x[0].shape

(784,)

In [18]:
W1.shape

(784, 50)

In [19]:
W2.shape

(50, 100)

In [20]:
W3.shape

(100, 10)

下面我们进行基于批处理的代码实现。

In [22]:
### 第一部分：初始化与批次设定
x,t = get_data()
network = init_network()

batch_size = 100 # 批数量
accuracy_cnt = 0

### 第二部分：分批大循环 (The Batch For-Loop)
for i in range(0,len(x),batch_size): # range(起始值,结束值,步长 == 批数量)
    # 也就是说，循环变量 i 不再是 0, 1, 2, 3 这样递增了，而是跳跃式的：第一次循环 i=0，第二次 i=100，第三次 i=200……以此类推。
    x_batch = x[i:i + batch_size] 
    # 利用 Python 的切片（Slicing）技术把图片分批切出来。
    # 当 i=0 时，x[0:100]，代表切出第 0 到 99 张图片，打包存进 x_batch。
    # 当 i=100 时，x[100:200]，代表切出第 100 到 199 张图片……
    # 此时 x_batch 的形状（Shape）是一个二维矩阵：(100, 784)。
    y_batch =predict(network,x_batch)
    # 把这 100 张图片组成的矩阵一次性送入神经网络！
    # 神经网络内部的矩阵乘法会同时计算这 100 张图。
    # 返回的 y_batch 是一个 (100, 10) 的大矩阵，里面包含了这 100 张图片分别对应 0~9 数字的概率。
    p = np.argmax(y_batch,axis = 1)
    # 现在的 y_batch 是个二维矩阵（有行有列）。axis=1 意思是：“在每一行(沿着第1维的方向)里分别寻找最大值的索引”。
    # 最终得到的 p 会是一个包含 100 个数字的一维数组，里面装的是这 100 张图片各自的预测答案。
    accuracy_cnt += np.sum(p == t[i:i+batch_size]) # 批量对答案，一次性算出对了几道题

### 第三部分：计算总分
print('Accuracy:' + ' ' + str(float(accuracy_cnt)/len(x)))

Accuracy: 0.9352


In [23]:
y = np.array([1,2,1,0])
t = np.array([1,2,0,0])
print(y==t)
print(np.sum(y==t))

[ True  True False  True]
3
